<a href="https://colab.research.google.com/github/LailaBulh/Ingenieria_de_Datos_Avanzada/blob/main/Data_Profiling_con_PySpark_LB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Data Profiling con PySpark**
**Nombre**: Laila Montserrat Bulhosen Ramos     **Matricula**: 263166

**Docente:** Dr. Vicente García Jiménez

**Materia:** Ingeniería de Datos Avanzada




**Link Github:** [Data_Profiling_con_PySpark](https://github.com/LailaBulh/Ingenieria_de_Datos_Avanzada/blob/main/Data_Profiling_con_PySpark_LB.ipynb)

**Fecha:** Mayo 2026

## **Instalación de PySpark**

In [1]:
try:
  import pyspark

  print('PySpark ya esta instalado')
  print(f'Version de PySpark instalada: {pyspark.__version__}')


except ModuleNotFoundError:
    print("PySpark no está instalado. Instalando...")
    !pip install pyspark -q

    import pyspark
    print("Instalación completada")
    print("Versión:", pyspark.__version__)

PySpark ya esta instalado
Version de PySpark instalada: 4.0.2


### **Creación de SparkSession**

In [2]:
from pyspark.sql import SparkSession

In [4]:
spark = (
    SparkSession.builder

    ### Nombre visible en la Spark UI y en los logs del clúster
    .appName('Data_Profiling')

    ### Número de particiones en operaciones de shuffle (default: 200)
    ### 200 particiones para datasets pequeños es excesivo y lento
    ### Regla general: 2-4 particiones por core en el clúster
    .config('spark.sql.shuffle.partitions', '8')

    ### Si ya existe una sesión activa, la reutiliza (no crea una nueva)
    .getOrCreate()
)

print(f'   SparkSession lista')
print(f'   Nombre app       : {spark.sparkContext.appName}')
print(f'   Master           : {spark.sparkContext.master}')
print(f'   Cores disponibles: {spark.sparkContext.defaultParallelism}')

   SparkSession lista
   Nombre app       : Data_Profiling
   Master           : local[*]
   Cores disponibles: 2


## **Cargar archivo CSV**

In [5]:
spark = SparkSession.builder \
.appName("NYC Taxi Profiling") \
.getOrCreate()

!wget -O taxi.csv "https://www.dropbox.com/scl/fi/ya6wwi1ouvu7b5ng00zu3/yellow_tripdata_2016-03.csv?rlkey=49gbpo35mmh7p2codjw4kcfd3&dl=1"

df = spark.read.csv("taxi.csv", header=True, inferSchema=True)

--2026-05-15 03:49:22--  https://www.dropbox.com/scl/fi/ya6wwi1ouvu7b5ng00zu3/yellow_tripdata_2016-03.csv?rlkey=49gbpo35mmh7p2codjw4kcfd3&dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.3.18, 2620:100:6018:18::a27d:312
Connecting to www.dropbox.com (www.dropbox.com)|162.125.3.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://uc8ef4a9c616444b2cafc52976ad.dl.dropboxusercontent.com/cd/0/inline/DAdLhe0cL1085VKjN6J54TNPI6d2PEIxYOIAwSr0RuN0NchfvWyfP2QdjCB53us3KXQkHSA4YbQuK2C5sgbGZz81OES5TbdUDWWpQsG8WdNR21IQT3gr1GWFi-RvUSh9w-Q/file?dl=1# [following]
--2026-05-15 03:49:22--  https://uc8ef4a9c616444b2cafc52976ad.dl.dropboxusercontent.com/cd/0/inline/DAdLhe0cL1085VKjN6J54TNPI6d2PEIxYOIAwSr0RuN0NchfvWyfP2QdjCB53us3KXQkHSA4YbQuK2C5sgbGZz81OES5TbdUDWWpQsG8WdNR21IQT3gr1GWFi-RvUSh9w-Q/file?dl=1
Resolving uc8ef4a9c616444b2cafc52976ad.dl.dropboxusercontent.com (uc8ef4a9c616444b2cafc52976ad.dl.dropboxusercontent.com)... 162.125.3.15, 2620:100:6018:15::

In [9]:
### Mostrar primeras filas

display(df.limit(5).toPandas())

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,1,2016-03-01,2016-03-01 00:07:55,1,2.50,-73.976746,40.765152,1,N,-74.004265,40.746128,1,9.0,0.5,0.5,2.05,0.00,0.3,12.35
1,1,2016-03-01,2016-03-01 00:11:06,1,2.90,-73.983482,40.767925,1,N,-74.005943,40.733166,1,11.0,0.5,0.5,3.05,0.00,0.3,15.35
2,2,2016-03-01,2016-03-01 00:31:06,2,19.98,-73.782021,40.644810,1,N,-73.974541,40.675770,1,54.5,0.5,0.5,8.00,0.00,0.3,63.80
3,2,2016-03-01,2016-03-01 00:00:00,3,10.78,-73.863419,40.769814,1,N,-73.969650,40.757767,1,31.5,0.0,0.5,3.78,5.54,0.3,41.62
4,2,2016-03-01,2016-03-01 00:00:00,5,30.43,-73.971741,40.792183,3,N,-74.177170,40.695053,1,98.0,0.0,0.0,0.00,15.50,0.3,113.80


## **Data Profiling**

* estructura del dataset (schema, numero de filas y columnas)
* estadisticos descriptivos (describe o summary)
* analisis de correlacion entre variables numericas



### Estructura del Dataset

In [12]:
### Esquema de dataframe

print('Esquema "NYC Taxi Profiling" en Spark:\n', df.printSchema())

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)

Esquema "NYC Taxi Profiling" en Spark:
 None


In [13]:
### Número de filas

rows = df.count()
print(f'El total de filas en el dataframe es: {rows}')

El total de filas en el dataframe es: 12210952


In [14]:
### Número de columnas

col = len(df.columns)
print(f'El total de columnas en el dataframe es: {col}')

El total de columnas en el dataframe es: 19


### Estadísticos descriptivos

In [16]:
df.describe().show()

+-------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+---------------------+-----------------+
|summary|          VendorID|   passenger_count|     trip_distance|   pickup_longitude|   pickup_latitude|        RatecodeID|store_and_fwd_flag|  dropoff_longitude|  dropoff_latitude|      payment_type|       fare_amount|              extra|            mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|     total_amount|
+-------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+---------------------

In [17]:
df.summary().show()

+-------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+---------------------+-----------------+
|summary|          VendorID|   passenger_count|     trip_distance|   pickup_longitude|   pickup_latitude|        RatecodeID|store_and_fwd_flag|  dropoff_longitude|  dropoff_latitude|      payment_type|       fare_amount|              extra|            mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|     total_amount|
+-------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+---------------------

### Análisis de correlación entre variables númericas

In [18]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

In [25]:
### Identificar columnas númericas

numeric_cols = [f.name for f in df.schema.fields if f.dataType.typeName() in ('integer', 'double', 'float')]

print("Columnas numéricas:", numeric_cols)

Columnas numéricas: ['VendorID', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'RatecodeID', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount']


In [26]:
### Convertir columnas a vector

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vector = assembler.transform(df).select("features")


In [27]:
corr_matrix = Correlation.corr(df_vector, 'features', 'pearson').collect()[0][0]
print('Matriz de correlación:\n', corr_matrix)

Matriz de correlación:
 DenseMatrix([[ 1.00000000e+00,  2.89536529e-01, -5.39540035e-04,
              -5.41952496e-02,  5.42515513e-02, -3.46840902e-03,
              -5.09008001e-02,  5.09246764e-02, -1.71675563e-02,
               4.75016067e-04, -2.28500825e-04,  2.84557214e-03,
               1.10104939e-02,  5.81241502e-03, -1.84346958e-02,
               7.57247750e-04],
             [ 2.89536529e-01,  1.00000000e+00, -1.39650120e-04,
              -1.75900123e-02,  1.75705756e-02, -6.43805938e-03,
              -1.60072450e-02,  1.59804676e-02,  1.31436056e-02,
               9.57239993e-04,  2.23861617e-03,  4.36052458e-03,
              -4.51195435e-03,  8.67811163e-03, -2.30505856e-03,
               9.91872460e-04],
             [-5.39540035e-04, -1.39650120e-04,  1.00000000e+00,
              -6.77620438e-05,  6.82388627e-05,  1.20139219e-04,
              -6.95165367e-05,  6.93028238e-05,  4.89430086e-04,
               3.86384387e-03,  6.90426704e-04, -2.93933593e-05,
  